In [1]:
import torch
import torch.nn as nn
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
training_data = datasets.CIFAR10(root='data', train=True, download=True, transform = transform_train)
test_data = datasets.CIFAR10(root='data', train = False, download=False, transform=transform_test)

100%|██████████| 170M/170M [00:03<00:00, 46.4MB/s]


In [4]:
train_loader=DataLoader(training_data, batch_size=64, shuffle=True)
test_loader= DataLoader(test_data, batch_size=64, shuffle=False)

In [5]:
class CIFAR(nn.Module):
    def __init__(self):
        super().__init__()
        self.CNN1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3)
        self.bn1 = nn.BatchNorm2d(32)
        self.CNN2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)
        self.bn2 = nn.BatchNorm2d(64)
        self.CNN3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3)
        self.bn3 = nn.BatchNorm2d(128)
        self.relu = nn.ReLU()
        self.mxp = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.lin1 = nn.Linear(2048, 1024)
        self.lin2 = nn.Linear(1024, 512)
        self.lin3 = nn.Linear(512, 10)

    def forward(self, x):
        x = self.CNN1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.mxp(x)
        x = self.CNN2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.mxp(x)
        x = self.CNN3(x)
        x = self.bn3(x)
        x = self.relu(x)
        x = self.flatten(x)
        x = self.lin1(x)
        x = self.relu(x)
        x = self.lin2(x)
        x = self.relu(x)
        x = self.lin3(x)
        return x

In [6]:
model = CIFAR()
model = nn.DataParallel(model)
model = model.to(device)
loss_fn=nn.CrossEntropyLoss()
optim = torch.optim.Adam(model.parameters(), lr=0.001)
num_epochs = 20
totaloss=0
for epoch in range(num_epochs):
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)  
        optim.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        totaloss += loss
        loss.backward()
        optim.step()
    print(f"Epoch {epoch+1} Loss: {totaloss.item()}")
    totaloss = 0

Epoch 1 Loss: 1102.6207275390625
Epoch 2 Loss: 836.4241943359375
Epoch 3 Loss: 732.7872924804688
Epoch 4 Loss: 666.5848388671875
Epoch 5 Loss: 620.3884887695312
Epoch 6 Loss: 585.1810302734375
Epoch 7 Loss: 555.139892578125
Epoch 8 Loss: 533.6801147460938
Epoch 9 Loss: 508.6694641113281
Epoch 10 Loss: 492.5502624511719
Epoch 11 Loss: 478.40191650390625
Epoch 12 Loss: 458.54296875
Epoch 13 Loss: 449.08526611328125
Epoch 14 Loss: 435.1929931640625
Epoch 15 Loss: 426.3008728027344
Epoch 16 Loss: 414.04888916015625
Epoch 17 Loss: 405.513427734375
Epoch 18 Loss: 400.4781188964844
Epoch 19 Loss: 388.14654541015625
Epoch 20 Loss: 378.38189697265625


In [7]:
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    print(f'Accuracy: {correct/total*100:.2f}%')

Accuracy: 80.86%


In [8]:
torch.save(model.state_dict(), 'CIFAR-10_model.pth')